# DeepCardio-RAG & Arthritis Analysis Dashboard 🚀
Run the complete backend and frontend UI directly from Google Colab.

## 1. Mount Google Drive and Navigate to Project
Upload the project folder to your Google Drive, then execute this cell to access the files.

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Change this path if you uploaded the folder elsewhere in your Drive
PROJECT_DIR = '/content/drive/MyDrive/sssss'

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
    print("Successfully navigated to project directory!")
else:
    print(f"Directory {PROJECT_DIR} not found. Please modify the path at the top of this cell.")

KeyboardInterrupt: 

## 2. Install Required Dependencies
Install all Python and Node.js dependencies for running the FastAPI application and exposing it to the web.

In [ ]:
!pip install -r requirements.txt
!pip install fpdf2
!npm install -g localtunnel

## 3. Start Milvus Vector DB & Expose via ngrok
Starts an embedded Milvus server on Colab and opens a **TCP tunnel** so your local machine can connect to it as a remote Milvus backend.

> **Pre-requisite:** Get a free ngrok auth token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) and paste it in the cell below.


In [ ]:
# Install Milvus standalone server and pyngrok
!pip install milvus pyngrok -q

from milvus import default_server

# Start Milvus (binds to port 19530 by default)
default_server.start()
print(f"Milvus server running on port {default_server.listen_port}")


In [ ]:
import json
from pyngrok import ngrok

# ── PASTE YOUR NGROK AUTH TOKEN BELOW ──────────────────────────────────────
# Free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "PASTE_YOUR_TOKEN_HERE"
# ───────────────────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a TCP tunnel to the Milvus port
tunnel = ngrok.connect(default_server.listen_port, "tcp")
url = tunnel.public_url  # e.g. tcp://0.tcp.ngrok.io:12345

host = url.replace("tcp://", "").split(":")[0]
port = url.replace("tcp://", "").split(":")[1]

# Write connection info to the project directory — syncs to Google Drive locally
tunnel_info = {"host": host, "port": port}
with open("milvus_tunnel.json", "w") as f:
    json.dump(tunnel_info, f, indent=2)

print("=" * 60)
print("  Milvus ngrok tunnel active!")
print(f"  Host : {host}")
print(f"  Port : {port}")
print("=" * 60)
print("On your LOCAL machine, run:")
print("  python connect_colab_milvus.py")
print("This reads milvus_tunnel.json from Drive and patches .env.")


## 3. Run the Dashboard 🌐
The cell below will start the backend server and provide you with a **Public URL**.

**Instructions:**
1. Note the **Localtunnel Password** printed by the cell (this is your Colab instance's IP address).
2. Click the `https://xxxx.loca.lt` link generated at the bottom.
3. Enter the password when prompted and click **Submit**.
4. **CRITICAL:** Append `/dashboard/` to the end of the URL in your browser's address bar (e.g., `https://xxxx.loca.lt/dashboard/`) and press Enter to see the UI!

In [ ]:
import subprocess
import urllib.request
import time

# Start the FastAPI server in the background
server_process = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])

time.sleep(3) # Wait for server to start

# Get public IP to use as the password to bypass Localtunnel's warning screen
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

print("="*60)
print(f"🔑 Your LocalTunnel Endpoint Password is: {external_ip}")
print("============================================================\n")

# Start LocalTunnel to expose port 8000
!lt --port 8000